## Gold Layer
###### banking.silver.fact_repayment -> banking.gold.fact_redelinquency_event + banking.gold.agg_redelinquency_branch_product

In [0]:
from pyspark.sql.functions import (col, lag, datediff, when, count, sum as _sum, avg,
                                   round as _round, current_timestamp)
from pyspark.sql.window import Window

In [0]:
fact = spark.read.table("banking.silver.fact_repayment")
display(fact)

In [0]:
# Same customer, same product, in due-date order

'''
The SQL equivalent of that PySpark window

SELECT
  *,
  LAG(cure_date) OVER (
    PARTITION BY customer_id, product_code
    ORDER BY due_date
  ) AS prev_cure_date
FROM banking.silver.fact_repayment
'''

w = Window.partitionBy("customer_id", "product_code").orderBy("due_date")

events = (
    fact
    .withColumn("prev_cure_date", lag("cure_date").over(w))
    .withColumn("days_since_last_cure", datediff(col("due_date"), col("prev_cure_date")))
    .withColumn(
        "redelinquency_30d_flag",
        when(col("days_since_last_cure").between(0, 30), 1).otherwise(0)
    )
)

display(events.select("customer_id", "product_code","due_date","cure_date", "prev_cure_date", "days_since_last_cure", "redelinquency_30d_flag"))

In [0]:
(events.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("banking.gold.fact_redelinquency_event"))

In [0]:
# KPI table: grain = branch x product
kpi = (
    events.groupBy("branch_id", "branch_name", "region",
                   "product_code", "product_desc", "product_type", "shariah_structure")
    .agg(
        count("*").alias("total_events"), # total repayment/event records in that group
        _sum("redelinquency_30d_flag").alias("total_redelinquencies"), # how many of those had redelinquency_30d_flag = 1
        _round(_sum("redelinquency_30d_flag") / count("*") * 100, 2).alias("redelinquency_rate"),
        _round(_sum("installment_amount_sar"), 2).alias("total_amount_sar"),
        _round(avg("installment_amount_sar"), 2).alias("avg_amount_sar"),
    )
    .withColumn("gold_load_timestamp", current_timestamp())
)

display(kpi)

In [0]:
(kpi.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("banking.gold.agg_redelinquency_branch_product"))

In [0]:
display(kpi.orderBy(col("redelinquency_rate").desc()))